> **Chapter 15, Part 0** | Engineering lens. **Focus:** why cron-plus-scripts fails in four predictable ways, and why the fix is to model a pipeline as a graph of assets rather than a sequence of tasks.

# From Cron to Asset Graphs

The repo has pipelines but no scheduler. Chapter 9 ships a dbt project, Chapter 10 builds a retrieval service, and the trading platform moves market data through Flask and Postgres. None of it answers the question every team hits in month two: what runs, in what order, when an upstream changes, and what happens when step four fails at 2am.

The default answer is cron plus a pile of scripts. It breaks in four predictable ways.

1. **No dependency awareness.** `0 2 * * *  python build_marts.py` runs at 2am whether or not the 1am ingest succeeded. The marts get built on yesterday's data and nobody notices for a week.
2. **No partial recovery.** One bad partition means rerun the whole pipeline from the top, because the scripts have no idea which days are already done.
3. **No lineage.** Ask "what feeds the executive dashboard" and the only honest answer is "read all forty scripts."
4. **No idempotency.** Rerun the ingest and you double-count, because the script appends instead of replacing.

Modern orchestrators (Dagster, Airflow, Prefect, Temporal) exist to fix exactly these four failures. This chapter teaches the idea the way the rest of the repo teaches its hard topics: build the apparatus from first principles in pure Python, connect it to real code in this repo, then map it onto the production tool.

## The shift: tasks to assets

The older mental model is **tasks**: a task is a thing you run. Airflow's original DAG is a graph of tasks. The newer mental model, the one Dagster is built on, is **assets**: an asset is a thing that exists, defined by what it depends on and how to compute it. You do not "run a task"; you "materialize an asset", which means producing the current version of it from its upstreams.

The asset framing is the one I use in this chapter because it reuses a vocabulary the repo already built. In Chapter 12 a lineage graph was a DAG of data objects with a structural blast radius. An asset graph is the same DAG, made executable. Materialize it in topological order and you have orchestration.

In [1]:
# An asset, at its smallest: a name, the upstreams it needs, and how to compute it.
from dataclasses import dataclass
from typing import Callable


@dataclass
class Asset:
    name: str
    deps: list          # names of upstream assets
    compute: Callable   # compute(inputs: dict) -> value, inputs keyed by dep name


# A four-asset pipeline: raw events -> a staged table -> daily marts -> a report.
raw = Asset("raw_events", [], lambda i: list(range(100)))
staged = Asset("stg_events", ["raw_events"], lambda i: [x for x in i["raw_events"] if x % 2 == 0])
marts = Asset("daily_marts", ["stg_events"], lambda i: {"rows": len(i["stg_events"])})
report = Asset("report", ["daily_marts"], lambda i: f"report: {i['daily_marts']['rows']} rows")

for a in (raw, staged, marts, report):
    deps = ", ".join(a.deps) if a.deps else "(none)"
    print(f"{a.name:14s} <- {deps}")

raw_events     <- (none)
stg_events     <- raw_events
daily_marts    <- stg_events
report         <- daily_marts


That print is already a lineage listing, the thing cron could never give you. Each asset names its upstreams, so the dependency structure lives in the data, not in forty scripts and one engineer's memory.

What it does not yet do is run anything in the right order, skip work already done, recover from a failure, or stay safe under a rerun. Those are the next four notebooks.

## The bounded claim

I am not arguing you should write your own orchestrator, that Dagster is the only right choice, or that asset-oriented beats task-oriented orchestration for every workload. The claim is narrower: orchestration is best understood as ordered, idempotent, observable materialization of an asset graph, and about 150 lines of pure Python make every core concept concrete and runnable on a laptop.

## The chapter spine

| Notebook | What it builds |
|---|---|
| 15.0 (this one) | The four failures of cron; the asset abstraction; the bounded claim. |
| 15.1 | `Asset` and `AssetGraph`; topological materialization; cycle detection. |
| 15.2 | Partitions and idempotent backfills. |
| 15.3 | Sensors on a simulated clock and freshness policies. |
| 15.4 | The repo's real dbt project, parsed into an asset graph. |
| 15.5 | Failure, retries, and the Chapter 12 blast radius. |
| 15.6 | Every concept mapped to the real Dagster API. |
| 15.7 | Capstone: the trading platform's data, orchestrated. |
| 15.8 | When the schedule lies: four failure modes. |

## Three audiences

- **The engineer with a cron pile** who has felt all four failures and wants the mental model that fixes them. By the end of 15.2 you will have idempotent backfills in under a hundred lines.
- **The dbt user** who runs `dbt build` by hand and wonders what a scheduler adds. By the end of 15.4 your own dbt models will be an asset graph.
- **The platform engineer** evaluating Dagster. By the end of 15.6 you will see exactly which of your toy concepts each Dagster primitive replaces, and by 15.8 where it still bites.